# Generate activation maps

**Purpose.** Estimate the image regions contributing to a trained classifier's predictions using saliency or activation-mapping methods.

**Recommended use.** Use after model validation to examine whether predictive evidence is spatially consistent with the intended biological phenotype.

**Primary outputs.** Per-image activation overlays and class-level aggregate maps.

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry point

This workflow calls the following public function:

- [`spacr.deep_spacr.generate_activation_map`](https://einarolafsson.github.io/spacr/api/spacr/deep_spacr/index.html#spacr.deep_spacr.generate_activation_map)

```python
generate_activation_map(settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.deep_spacr import generate_activation_map

## 3. Settings and API reference

Review the parameter definitions here, then edit the values in the categorized code cells below. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the installed spaCR version so that the notebook remains aligned with the public API.

### [`spacr.deep_spacr.generate_activation_map`](https://einarolafsson.github.io/spacr/api/spacr/deep_spacr/index.html#spacr.deep_spacr.generate_activation_map)


#### Paths

- **`dataset`** *(optional)* — (str) - Path to the .tar archive of single-object PNG crops produced by generate_dataset, which the activation-map step opens with TarImageDataset. The plate folder is inferred two levels above it and CAM outputs are written next to it under &lt;tar_name&gt;/&lt;cam_type&gt;/. Must be a full path, not just a file name. Default ''.
- **`model_path`** *(optional)* — (str) - Path to a trained spaCR classifier saved as a whole PyTorch object (loaded with torch.load(weights_only=False), not a state_dict). Used when applying a model to a dataset tar and when generating activation maps. deep_spacr overwrites it with the freshly trained model whenever train is True, so set it only to score with an existing model. Default ''.

#### General

- **`channels`** *(optional)* — (list of int) - Zero-indexed image channels kept in merged/*.npy and measured by measure_crop; each entry produces its own &lt;object&gt;_channel_&lt;n&gt;_* intensity columns. The list length fixes where masks land, so cell/nucleus/pathogen_mask_dim must shift if you change it. Preprocessing silently resets it to range(n) when it does not match the number of channel folders found. Default [0,1,2,3].
- **`normalize`** *(optional)* — (bool) - Percentile-normalize each image channel (2nd to 98th percentile, clipped to 0-1) before display or model input; in the activation-map tool this rescales the image the CAM/saliency heatmap is drawn over. Turn it on when raw channels are too dim to read under the overlay. Affects display and input scaling only, never stored pixels. Default True.
- **`plot`** *(optional)* — (bool) - Render and save QC figures while the pipeline runs: channel montages and Cellpose mask overlays during segmentation, before/after filtration views and crop grids during measurement. It adds figures per batch, so a full plate becomes much slower and more memory-hungry; keep it for small or test_mode runs, which force it on. Default False.

#### Measurements

- **`manders_thresholds`** *(optional)* — (list) - Percentiles (0-100) at which Manders' overlap coefficients are computed. For each object, each entry thresholds both channels at that percentile; pixels above both count as overlap, and M1/M2 report each channel's fraction of total object intensity there, saved as M1_correlation_&lt;t&gt; and M2_correlation_&lt;t&gt;. High values isolate the brightest puncta. Requires calculate_correlation. Default [15, 85, 95].

#### Computer Vision Data Source

- **`image_size`** *(optional)* — (int) - Side length in pixels of the centre crop taken from each object PNG before it reaches the model. Images are cropped, not rescaled, so a larger value zero-pads and a smaller one throws away the object's edges. It is also the resolution the backbone is built at, which matters for ViT/Swin/inception. Match it to the crop size used when the dataset was generated. Default 224.

#### Computer Vision Model

- **`model_type`** *(optional)* — (str) - Backbone architecture for the single-object image classifier: any TorchVision classification model name (resnet50, maxvit_t, densenet121, ...). An unrecognised name is NOT fatal when it is read -- choose_model prints 'Invalid model_type' and returns None, and training fails afterwards -- and the special name 'custom' passes the name check then raises NotImplementedError. Bigger backbones need more memory and more labelled crops to beat a smaller one. Default 'maxvit_t'.

#### Activation Maps

- **`smoothgrad_samples`** *(optional)* — (int) - Noisy copies of the image averaged into one attribution map. A single map is dominated by the gradient's local jitter, so 8-50 samples smooth it into something stable enough to compare between images; 0 (the default) runs the method once and is what you want while you are still choosing a method, since it costs one forward-backward pass instead of N. Applies to every method, including the CAM family, where it is averaged explicitly rather than through captum. Default 0.
- **`smoothgrad_sigma`** *(optional)* — (float) - Standard deviation of the noise SmoothGrad adds, as a fraction of the image's intensity range. Too small and every sample is the same map, so averaging changes nothing; too large and the samples are of images the model has never seen, so the average describes the model's behaviour on noise rather than on your data. 0.1-0.2 is the usual band. Ignored when smoothgrad_samples is 0. Default 0.15.
- **`occlusion_window`** *(optional)* — (int) - Side length in pixels of the patch occlusion slides over the image, blanking it and recording how far the model's score falls. Larger windows are faster and blurrier and will miss a feature smaller than the window; smaller ones resolve fine structure at quadratically more forward passes. Occlusion is the only method here that needs no gradients at all, which is why it is worth its cost as a cross-check on the gradient family. Default 8.
- **`occlusion_stride`** *(optional)* — (int) - How far the occlusion patch moves between evaluations. Equal to occlusion_window it tiles without overlap and is fastest; half of it doubles the passes and halves the blockiness. A stride larger than the window leaves unmeasured gaps that appear as an artificial grid in the map. Default 4.
- **`ig_steps`** *(optional)* — (int) - Interpolation steps between the baseline and your image for integrated gradients. The method's guarantee - that the attributions sum to the score difference - only holds in the limit, so too few steps silently breaks it; 50 is the usual default and the completeness error is worth checking if you lower it. Cost is linear in this number. Default 50.
- **`ig_baseline`** *(optional)* — (str) - The 'absence of signal' image integrated gradients integrates away from: 'zero' is black, 'blur' is your own image blurred, 'noise' is random. This choice IS the explanation's reference point and changes the result - a black baseline attributes to everything bright, which on dark-field microscopy means it attributes to the object merely for existing. 'blur' keeps the low-frequency content and asks what the detail contributes. Default 'zero'.
- **`attribution_steps`** *(optional)* — (int) - Points along the deletion and insertion curves used to score a map. At each step the highest-ranked remaining pixels are removed (or added) and the model re-run, so this is the resolution of the area-under-curve that judges whether the map describes what the model actually uses. More steps give a smoother AUC at linearly more forward passes. Default 12.
- **`attribution_baseline`** *(optional)* — (str) - What a pixel is replaced with when the deletion/insertion curves remove it: 'blur', 'zero' or 'noise'. It is a confound, not a detail - blanking to zero creates a hard edge the model has never seen, so part of the score drop measures the artefact rather than the lost information. 'blur' is the least out-of-distribution and is the default; comparing two baselines is a fair way to ask how much of your AUC is real. Default 'blur'.
- **`sanity_check`** *(optional)* — (bool) - Randomise the model's weights layer by layer and re-attribute, then report how similar the map stays. A method that produces nearly the same picture for a randomised model is an edge detector, not an explanation - and measured on a small CNN the whole CAM family, including the Grad-CAM spaCR defaults to, fails this while saliency and integrated gradients pass. The number is reported for YOUR model rather than assumed, which is the point. Costs one extra attribution per randomised layer. Default True.
- **`object_type`** *(optional)* — (str) - Which mask decides where an object is when the pointing game scores an attribution map: 'cell', 'nucleus', 'pathogen' or 'cytoplasm'. The pointing game asks only whether the map's single hottest pixel lands inside that mask, so it is cheap and coarse - it says nothing about the rest of the map, and a method can score 1.0 while attributing nonsense everywhere else. Default 'cell'.
- **`cam_type`** *(optional)* — (str) - Which attribution map is computed. 'gradcam' weights the target_layer feature maps by their pooled gradients into a coarse heatmap of the region that drove the call; 'gradcam_pp' currently computes the identical map and only changes the output folder and table name. 'saliency_image' sums the absolute input gradient into one map; 'saliency_channel' keeps it per channel so you can see which stain mattered. Default 'gradcam'.
- **`target_layer`** *(optional)* — (str) - Dotted attribute path to the convolutional layer whose activations and gradients Grad-CAM hooks, e.g. 'base_model.blocks.3.layers.1.layers.MBconv.layers.conv_b'; utils.recommend_target_layers(model) lists valid names. Later layers give class-specific but coarse maps, earlier ones finer detail. Required for 'gradcam'/'gradcam_pp' - it is auto-filled only when model_type is exactly 'maxvit', and left None it raises. Default None.
- **`overlay`** *(optional)* — (bool) - In the batch-grid figures, draw the activation map in the 'jet' colormap at 50 percent alpha over the source image. Turn it off and the grid tiles are left empty apart from the predicted-class label, so keep it on whenever plot is enabled. It never affects the per-object activation PNGs saved to disk, which are always the bare map. Default True.
- **`correlation`** *(optional)* — (bool) - Correlate every input channel against every activation-map channel per image and write the result to the &lt;cam_type&gt;_correlations table: a Pearson coefficient plus Manders M1/M2 at each manders_thresholds percentile (15, 50, 75 by default). Use it to quantify which stain the model attends to instead of eyeballing heatmaps; it needs save=True to reach the database. Default True.
- **`normalize_input`** *(optional)* — (bool) - Apply the same per-channel mean=0.5, std=0.5 normalisation used during training to each image before it enters the model when generating activation maps. Keep it matched to how the model was trained, otherwise inputs are off-distribution and both the predicted classes and the maps are meaningless. Distinct from 'normalize', which only percentile-stretches images for display. Default True.

#### Advanced

- **`n_jobs`** *(optional)* — (int) - CPU workers for parallel stages: measurement, mask adjustment, DataLoader loading, and the sklearn/UMAP calls where -1 means every core. Raise it to shorten CPU-bound steps until RAM or disk I/O saturates. Note the measure-and-crop pipeline overrides your value with cpu_count()-4. Defaults vary by pipeline: cpu_count()-4, -1, or None.
- **`batch_size`** *(optional)* — (int) - How many images are held and processed together in one pass: field stacks during normalization and Cellpose segmentation, crops per step during classifier training and activation maps. Raising it speeds runs up but increases RAM/VRAM roughly linearly; lower it on out-of-memory errors. Defaults: 50 for mask generation, 64 for training.
- **`shuffle`** *(optional)* — (bool) - Shuffle the tar dataset in the DataLoader when generating activation maps, so each batch-grid PDF shows a mixed sample rather than consecutive files from one plate or class. Set False for a deterministic, file-order pass you can line up against the dataset listing. Default True.
- **`save`** *(optional)* — (bool or list of bool) - Whether to save masks to disk. Can be a list of three booleans for [cell, nucleus, pathogen] independently. Default varies by module -- False for most, True for the sequencing and regression paths.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    # Paths
    # Optional settings
    'dataset': 'path',
    'model_path': 'path',

    # General
    # Optional settings
    'channels': [1, 2, 3],
    'normalize': True,
    'plot': False,

    # Measurements
    # Optional settings
    'manders_thresholds': [15, 50, 75],

    # Computer Vision Data Source
    # Optional settings
    'image_size': 224,

    # Computer Vision Model
    # Optional settings
    'model_type': 'maxvit',

    # Activation Maps
    # Optional settings
    'smoothgrad_samples': 0,
    'smoothgrad_sigma': 0.15,
    'occlusion_window': 8,
    'occlusion_stride': 4,
    'ig_steps': 50,
    'ig_baseline': 'zero',
    'attribution_steps': 12,
    'attribution_baseline': 'blur',
    'sanity_check': True,
    'object_type': 'cell',
    'cam_type': 'gradcam',
    'target_layer': None,
    'overlay': True,
    'correlation': True,
    'normalize_input': True,

    # Advanced
    # Optional settings
    'n_jobs': None,
    'batch_size': 64,
    'shuffle': True,
    'save': True,
}

In [ ]:
generate_activation_map(settings)

## Outputs and next steps

Per-image activation overlays and class-level aggregate maps.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)